In [ ]:
from dataclasses import asdict
from datetime import datetime, timedelta

import pandas as pd

from air_quality_monitor.features import FeatureEngineer
from air_quality_monitor.models import AirQualityReading, HourlyForecast


In [ ]:
# Sample AQI reading template
sample_aqr = AirQualityReading(
    city="Sarajevo",
    state="Federation of B&H",
    country="Bosnia Herzegovina",
    timezone="Europe/Sarajevo",
    latitude=43.8519774,
    longitude=18.3866868,
    aqi=63,
    main_pollutant="p2",
    pollutant_timestamp=datetime.fromisoformat("2026-03-19 16:00:00+00:00"),
    temperature=7,
    humidity=47,
    pressure=1016,
    wind_speed=4.17,
    wind_direction=60,
    heat_index=5,
    weather_timestamp=datetime.fromisoformat("2026-03-19 16:00:00+00:00"),
    collected_at=datetime.fromisoformat("2026-03-19 17:10:42.923474"),
)

In [ ]:
# Sample hourly forecast template
sample_hf = HourlyForecast(
    city="Sarajevo",
    state="Federation of B&H",
    country="Bosnia Herzegovina",
    timezone="Europe/Sarajevo",
    latitude=43.8519774,
    longitude=18.3866868,
    forecast_for=datetime.fromisoformat("2026-03-20 21:00:00"),
    temperature=4.36,
    feels_like=2.08,
    pressure=1021,
    humidity=72,
    dew_point=-0.21,
    uvi=0,
    clouds=0,
    visibility=10000,
    wind_speed=2.57,
    wind_direction=210,
    wind_gust=6.83,
    pop=0.0,
    rain=2.47,
    snow=4.28,
    weather_main="Clear",
    weather_desc="clear sky",
    collected_at=datetime.fromisoformat("2026-03-20 21:33:45.862527"),
)

In [ ]:
# Build AQI DataFrame
base_time = datetime(2026, 3, 20, 21, 33, 46)
rows = []
for hour in range(72):
    for city in ["Sarajevo", "London"]:
        row = asdict(sample_aqr)
        row["id_aqi"] = 0
        row["city"] = city
        row["pollutant_timestamp"] = base_time + timedelta(hours=hour)
        row["aqi"] = hour + (10 if city == "London" else 0)
        rows.append(row)

sample_aqi_df = pd.DataFrame(rows)
print(f"AQI DataFrame: {len(sample_aqi_df)} rows")
sample_aqi_df.head()

In [ ]:
# Build Weather DataFrame
horizons = [0, 1, 2, 4, 8, 12, 24, 48]
rows = []

for hour in range(72):
    collected_at = base_time + timedelta(hours=hour)
    for city in ["Sarajevo", "London"]:
        for h in horizons:
            row = asdict(sample_hf)
            row["id_we"] = 0
            row["city"] = city
            row["collected_at"] = collected_at
            row["forecast_for"] = collected_at + timedelta(hours=h)
            rows.append(row)

sample_hourly_df = pd.DataFrame(rows)
print(f"Weather DataFrame: {len(sample_hourly_df)} rows")
sample_hourly_df.head(10)

In [ ]:
# Check forecast_for range
print("Weather forecast_for range:")
print(f"  Min: {sample_hourly_df['forecast_for'].min()}")
print(f"  Max: {sample_hourly_df['forecast_for'].max()}")
print()
print("AQI pollutant_timestamp range:")
print(f"  Min: {sample_aqi_df['pollutant_timestamp'].min()}")
print(f"  Max: {sample_aqi_df['pollutant_timestamp'].max()}")

In [ ]:
# Mock storage class
class MockStorage:
    def __init__(self, df: pd.DataFrame):
        self.df = df
    
    def read(self) -> pd.DataFrame:
        return self.df

In [ ]:
# Run FeatureEngineer.build()
aqi_storage = MockStorage(sample_aqi_df)
weather_storage = MockStorage(sample_hourly_df)
engineer = FeatureEngineer(aqi_storage, weather_storage)
result_df = engineer.build()

print(f"Result DataFrame: {len(result_df)} rows")
print(f"Columns: {list(result_df.columns)}")

In [ ]:
# Explore the result
result_df.head(20)

In [ ]:
# Check lag features for a specific row
# For Sarajevo at hour 10, aqi should be 10, lag_1h should be 9, etc.
result_df[result_df['city_Sarajevo'] == True][['aqi', 'aqi_current', 'aqi_lag_1h', 'aqi_lag_2h', 'aqi_lag_4h', 'horizon']].head(20)

In [ ]:
# Check unique horizons
print("Unique horizons:", sorted(result_df['horizon'].unique()))
print(f"Row count:\t{len(result_df)}")

In [ ]:
filtered = result_df[(result_df["city_Sarajevo"] == True) & (result_df["horizon"] == 0)]
filtered = filtered.sort_values("collected_at")
row = filtered.iloc[50]

print(filtered.iloc[50]["collected_at"], filtered.iloc[50]["hour"])
print(row)

In [ ]:
len(result_df[(result_df["city_Sarajevo"] == True) & (result_df["horizon"] == 0)])